In [1]:
from flask import Flask, render_template, jsonify, request
import math

app = Flask(__name__)

# Initialize 4 objects with unique IDs
def create_object(id):
    return {
        "id": id,
        "x": (id * 40) - 60, "y": 0.0,  # Offset starting positions
        "vx": 0.0, "vy": 0.0,
        "ix": 0.0, "iy": 0.0,          # Integrals
        "lx": 0.0, "ly": 0.0           # Last errors
    }

objects = [create_object(i) for i in range(4)]
state = {"target_x": 0.0, "target_y": 0.0, "mass": 1.2, "damping": 0.98}

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/update', methods=['POST'])
def update():
    data = request.json
    kp = float(data.get('kp', 10.7)) / 10.0
    kd = float(data.get('kd', 7.3)) / 10.0
    ki = float(data.get('ki', 3.0)) / 100.0
    
    disturbances = data.get('disturbances', []) # List of {x, y} for each obj

    output = []
    for i, obj in enumerate(objects):
        d = disturbances[i]
        
        # PID Logic per object
        ex = state["target_x"] - obj["x"]
        ey = state["target_y"] - obj["y"]
        
        obj["ix"] += ex
        obj["iy"] += ey
        
        dx = ex - obj["lx"]
        dy = ey - obj["ly"]
        
        # Control Force
        fcx = (ex * kp) + (obj["ix"] * ki) + (dx * kd)
        fcy = (ey * kp) + (obj["iy"] * ki) + (dy * kd)
        
        # Physics
        obj["vx"] = (obj["vx"] + (fcx + d['x']) / state["mass"]) * state["damping"]
        obj["vy"] = (obj["vy"] + (fcy + d['y']) / state["mass"]) * state["damping"]
        obj["x"] += obj["vx"]
        obj["y"] += obj["vy"]
        
        obj["lx"], obj["ly"] = ex, ey
        
        output.append({"id": i, "x": obj["x"], "y": obj["y"]})

    return jsonify(output)

if __name__ == '__main__':
    app.run(debug=True, port=8011, use_reloader = False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:8011
Press CTRL+C to quit


In [ ]:
from flask import Flask, render_template, jsonify, request
import math

app = Flask(__name__)

# formation spacing (pixels between units)
SPACING = 120 

def create_object(id):
    # Calculate a unique station for each object in a row
    # Centers the row by offsetting based on the total count
    station_x = (id * SPACING) - ((4 - 1) * SPACING / 2)
    return {
        "id": id,
        "x": station_x, "y": -150.0,    # Start them "above" their stations
        "target_x": station_x,          # Each has a unique horizontal target
        "target_y": 0.0,
        "vx": 0.0, "vy": 0.0,
        "ix": 0.0, "iy": 0.0,
        "lx": 0.0, "ly": 0.0
    }

objects = [create_object(i) for i in range(4)]
state = {"mass": 1.2, "damping": 0.98}

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/update', methods=['POST'])
def update():
    data = request.json
    kp = float(data.get('kp', 10.7)) / 10.0
    kd = float(data.get('kd', 7.3)) / 10.0
    ki = float(data.get('ki', 3.0)) / 100.0
    disturbances = data.get('disturbances', [])

    output = []
    for i, obj in enumerate(objects):
        d = disturbances[i]
        
        # Error relative to their specific Station
        ex = obj["target_x"] - obj["x"]
        ey = obj["target_y"] - obj["y"]
        
        obj["ix"] += ex
        obj["iy"] += ey
        dx = ex - obj["lx"]
        dy = ey - obj["ly"]
        
        # Individual Control Force
        fcx = (ex * kp) + (obj["ix"] * ki) + (dx * kd)
        fcy = (ey * ey * ki if ey > 0 else ey * kp) # Simplified for vertical pull
        # Re-applying standard PID for simplicity:
        fcy = (ey * kp) + (obj["iy"] * ki) + (dy * kd)
        
        obj["vx"] = (obj["vx"] + (fcx + d['x']) / state["mass"]) * state["damping"]
        obj["vy"] = (obj["vy"] + (fcy + d['y']) / state["mass"]) * state["damping"]
        
        obj["x"] += obj["vx"]
        obj["y"] += obj["vy"]
        obj["lx"], obj["ly"] = ex, ey
        
        output.append({"id": i, "x": obj["x"], "y": obj["y"]})

    return jsonify(output)

if __name__ == '__main__':
    app.run(debug=True, port=8011, use_reloader = False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:8011
Press CTRL+C to quit
127.0.0.1 - - [13/Apr/2026 14:56:20] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:56:20] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:56:20] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:56:20] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:56:20] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:56:20] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:56:20] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:56:20] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:56:20] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:56:20] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:56:20] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:56:20] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:56:20] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:56:20] "POST /update HTTP/1.1" 200 -
127.0.0.1 - 